# 01 — Feature Engineering: LaLiga *Hypermotion*

Notebook dedicado a la construcción del conjunto final de variables que alimentará los modelos del trabajo. Partiendo de los DataFrames limpios persistidos por el notebook `00_eda.ipynb`, se generan las *features* pre-partido siguiendo dos reglas innegociables:

- **Anti-leakage temporal**: toda variable se calcula exclusivamente con información disponible antes del *kickoff*. Nunca se utilizan datos del propio partido ni de jornadas posteriores.
- **Cronología por equipo**: las ventanas *rolling* se calculan sobre la secuencia real de partidos de cada club, no sobre la jornada oficial del calendario, para preservar la integridad temporal frente a aplazamientos.

El resultado final es un *dataset* tabular, una fila por partido, con todas las variables predictoras y la variable objetivo, listo para ser consumido por los modelos.

## 0. Setup

### 0.1. Carga de dependencias, definición de parámetros del *feature engineering* y lectura de los DataFrames.

In [24]:
from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

In [25]:
ROOT = Path.cwd()

INTERIM_DIR   = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    "matches_clean.parquet",
    "standings_clean.parquet",
    "squads_clean.parquet",
    "clubs_clean.parquet",
]

for f in required_files:
    path = INTERIM_DIR / f
    assert path.exists(), f"Falta el archivo {path}. Ejecuta primero 00_eda.ipynb"
    
print("\n✓ Todos los archivos de entrada disponibles")


✓ Todos los archivos de entrada disponibles


In [26]:
ROLLING_WINDOWS = [3, 5, 10]

ELO_INITIAL        = 1500
ELO_K              = 20
ELO_HOME_ADVANTAGE = 65

TEMPORADA_ACTUAL = 2025

SPLITS = {
    "train":      list(range(2010, 2020)),
    "validation": [2020, 2021, 2022],
    "test":       [2023, 2024],
    "demo":       [2025],
}

### 0.2. Carga de datos desde `data/interim/`

Lectura de los cuatro DataFrames limpios persistidos por el notebook 00.

In [27]:
matches   = pd.read_parquet(INTERIM_DIR / "matches_clean.parquet")
standings = pd.read_parquet(INTERIM_DIR / "standings_clean.parquet")
squads    = pd.read_parquet(INTERIM_DIR / "squads_clean.parquet")
clubs     = pd.read_parquet(INTERIM_DIR / "clubs_clean.parquet")

matches = matches.sort_values(["season", "date"]).reset_index(drop=True)

print(f"{'DataFrame':<12} {'Filas':>10} {'Columnas':>10}")
print(f"{'-'*34}")
print(f"{'matches':<12} {len(matches):>10,} {len(matches.columns):>10}")
print(f"{'standings':<12} {len(standings):>10,} {len(standings.columns):>10}")
print(f"{'squads':<12} {len(squads):>10,} {len(squads.columns):>10}")
print(f"{'clubs':<12} {len(clubs):>10,} {len(clubs.columns):>10}")

DataFrame         Filas   Columnas
----------------------------------
matches           7,295         38
standings        14,784         12
squads           10,461         10
clubs               352          5


## 1. Familia A — Forma reciente

Variables que capturan el estado de forma reciente de cada equipo justo antes del partido. Se calculan tres métricas (puntos por partido (PPG), goles a favor, goles en contra) sobre tres ventanas (N = 3, 5, 10 partidos), lo que produce nueve variables por equipo y dieciocho por partido.

El procedimiento sigue el patrón estándar para datos de partidos:

1. Se transforma el dataset de partidos de formato ancho (una fila con local y visitante) a formato largo (una fila por equipo y partido).
2. Sobre el formato largo, se ordena por `(equipo, temporada, fecha)` y se calculan las métricas móviles con `.shift(1)`, lo que **excluye estrictamente el partido en curso** de su propia ventana.
3. Las features resultantes se reincorporan a `matches` mediante un merge con prefijos `home_` y `away_`.

In [28]:
def matches_to_long(matches: pd.DataFrame) -> pd.DataFrame:
    home = matches[['season', 'date', 'jornada', 'home_team', 'fthg', 'ftag', 'ftr']].copy()
    home.columns = ['season', 'date', 'jornada', 'team', 'gf', 'gc', 'ftr']
    home['is_home'] = True
    home['pts'] = home['ftr'].map({'H': 3, 'D': 1, 'A': 0})
    
    away = matches[['season', 'date', 'jornada', 'away_team', 'ftag', 'fthg', 'ftr']].copy()
    away.columns = ['season', 'date', 'jornada', 'team', 'gf', 'gc', 'ftr']
    away['is_home'] = False
    away['pts'] = away['ftr'].map({'H': 0, 'D': 1, 'A': 3})
    
    matches_long = pd.concat([home, away], ignore_index=True)
    matches_long = matches_long.sort_values(['team', 'season', 'date']).reset_index(drop=True)
    
    return matches_long


matches_long = matches_to_long(matches)

print(f"Formato largo: {len(matches_long):,} filas")
print(f"Equipos: {matches_long['team'].nunique()}")

Formato largo: 14,590 filas
Equipos: 68


In [29]:
def add_rolling_features(matches_long: pd.DataFrame, window: int) -> pd.DataFrame:
    
    grouped = matches_long.groupby(['team', 'season'], group_keys=False)
    
    matches_long[f'ppg_{window}'] = grouped['pts'].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).mean()
    )
    matches_long[f'gf_{window}'] = grouped['gf'].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).sum()
    )
    matches_long[f'gc_{window}'] = grouped['gc'].transform(
        lambda s: s.shift(1).rolling(window, min_periods=1).sum()
    )
    
    return matches_long


for w in ROLLING_WINDOWS:
    matches_long = add_rolling_features(matches_long, w)

feature_cols_rolling = [c for c in matches_long.columns 
                        if any(c.startswith(p) for p in ['ppg_', 'gf_', 'gc_'])]
print(f"Features rolling generadas: {feature_cols_rolling}")

Features rolling generadas: ['ppg_3', 'gf_3', 'gc_3', 'ppg_5', 'gf_5', 'gc_5', 'ppg_10', 'gf_10', 'gc_10']


In [30]:
home_view = matches_long[matches_long['is_home']][
    ['season', 'date', 'team'] + feature_cols_rolling
].copy()
home_view = home_view.rename(columns={
    'team': 'home_team',
    **{c: f'home_{c}' for c in feature_cols_rolling}
})

away_view = matches_long[~matches_long['is_home']][
    ['season', 'date', 'team'] + feature_cols_rolling
].copy()
away_view = away_view.rename(columns={
    'team': 'away_team',
    **{c: f'away_{c}' for c in feature_cols_rolling}
})

matches_fa = matches.merge(home_view, on=['season', 'date', 'home_team'], how='left')
matches_fa = matches_fa.merge(away_view, on=['season', 'date', 'away_team'], how='left')

### 1.3. Validación anti-leakage

Se realizan tres comprobaciones para certificar que las features rolling no contienen información del propio partido:

1. Primer partido de cada equipo en cada temporada: las features rolling deben ser NaN.
2. Caso concreto reproducible con un cálculo manual de `home_ppg_5` para un partido específico y comparación con el valor automático.

In [31]:
test1_passed = True
primer_partido_por_temporada = (
    matches_long
    .sort_values('date')
    .groupby(['team', 'season'])
    .head(1)
)

violaciones_ppg_5 = primer_partido_por_temporada['ppg_5'].notna().sum()
total_primeros = len(primer_partido_por_temporada)

print(f"Test 1: {'✓ pasa' if violaciones_ppg_5 == 0 else '✗ falla'}")


print(f"\nTest 2 - Verificación manual sobre Valladolid 2010-11:")
vld_long = matches_long[(matches_long['team'] == 'Valladolid') & (matches_long['season'] == 2010)].head(6)
print(vld_long[['date', 'pts', 'ppg_5']].to_string(index=False))
print(f"\n  Partido 1: ppg_5 = NaN (sin historia)        → ✓ correcto")
print(f"  Partido 2: ppg_5 = pts_partido_1                → ¿coincide?")
print(f"  Partido 3: ppg_5 = mean(pts_1, pts_2)           → ¿coincide?")
print(f"  Partido 6: ppg_5 = mean(pts_1..pts_5)           → ¿coincide?")

Test 1: ✓ pasa

Test 2 - Verificación manual sobre Valladolid 2010-11:
      date pts  ppg_5
2010-08-27   3    NaN
2010-09-05   3   3.00
2010-09-12   3   3.00
2010-09-19   0   3.00
2010-09-26   1   2.25
2010-10-04   1   2.00

  Partido 1: ppg_5 = NaN (sin historia)        → ✓ correcto
  Partido 2: ppg_5 = pts_partido_1                → ¿coincide?
  Partido 3: ppg_5 = mean(pts_1, pts_2)           → ¿coincide?
  Partido 6: ppg_5 = mean(pts_1..pts_5)           → ¿coincide?


## 2. Familia B — Datos pre-partido

Variables que capturan la situación clasificatoria de los dos equipos justo antes del partido. Se construyen seis features:

- `home_pts_pre`, `away_pts_pre`: puntos acumulados al cierre de la jornada anterior.
- `home_gd_pre`, `away_gd_pre`: diferencia de goles acumulada al cierre de la jornada anterior.

In [32]:
standings_pre = standings.copy()
standings_pre['jornada_target'] = standings_pre['jornada'] + 1

standings_pre = standings_pre.rename(columns={
    'equipo_canonical': 'team',
    'pts': 'pts_pre',
    'dif_goles': 'gd_pre',
})[['season', 'jornada_target', 'team', 'pts_pre', 'gd_pre']]

In [33]:
home_view_b = standings_pre.rename(columns={
    'team': 'home_team',
    'pts_pre': 'home_pts_pre',
    'gd_pre':  'home_gd_pre',
})

away_view_b = standings_pre.rename(columns={
    'team': 'away_team',
    'pts_pre': 'away_pts_pre',
    'gd_pre':  'away_gd_pre',
})

matches_fb = matches_fa.merge(
    home_view_b,
    left_on=['season', 'jornada', 'home_team'],
    right_on=['season', 'jornada_target', 'home_team'],
    how='left'
).drop(columns=['jornada_target'])

matches_fb = matches_fb.merge(
    away_view_b,
    left_on=['season', 'jornada', 'away_team'],
    right_on=['season', 'jornada_target', 'away_team'],
    how='left'
).drop(columns=['jornada_target'])

assert len(matches_fb) == len(matches_fa), "ALERTA: el merge ha duplicado filas"

In [34]:
matches_fb['pts_diff'] = matches_fb['home_pts_pre'] - matches_fb['away_pts_pre']
matches_fb['gd_diff']  = matches_fb['home_gd_pre']  - matches_fb['away_gd_pre']
matches_fb['pos_diff'] = matches_fb['home_pos_pre'] - matches_fb['away_pos_pre']

## 3. Familia C — Valor de mercado

Las tres variables de esta familia (`home_mv`, `away_mv`, `mv_diff`) ya están calculadas en el dataset cargado. No se realiza ningún cálculo adicional en este notebook.

In [35]:
print(matches_fb[['home_mv', 'away_mv', 'mv_diff']].describe().round(2).to_string())

       home_mv  away_mv  mv_diff
count   7295.0   7295.0   7295.0
mean     20.86    20.86     -0.0
std      12.32    12.31    16.47
min        5.4      5.4   -72.85
25%       12.4     12.4    -7.76
50%      16.88    16.88      0.0
75%      25.55    25.55      7.7
max       80.0     80.0    72.85


## 4. Familia D — Sistema Elo

Sistema de rating dinámico que actualiza la fuerza estimada de cada equipo tras cada partido en función del resultado obtenido y de las expectativas previas. Los detalles de implementación son los siguientes:

- **Rating inicial**: 1.500 puntos para todos los equipos.
- **Factor K**: 20, controla la magnitud del ajuste tras cada partido.
- **Ventaja local**: 65 puntos, sumados implícitamente al rating del local al calcular la probabilidad esperada.
- **Reset por temporada**: cada temporada arranca con todos los equipos en 1.500 puntos.

Se generan tres variables: `home_elo_pre`, `away_elo_pre` y `elo_diff`.

In [36]:
def expected_score(rating_home: float, rating_away: float, home_advantage: float) -> float:
    diff = rating_home + home_advantage - rating_away
    return 1.0 / (1.0 + 10 ** (-diff / 400))


def margin_multiplier(goal_diff: int) -> float:
    g = abs(goal_diff)
    if g <= 1:
        return 1.0
    elif g == 2:
        return 1.5
    elif g == 3:
        return 1.75
    else:
        return (11 + g) / 8.0


def update_elo(rating: float, actual: float, expected: float, k: float, multiplier: float = 1.0) -> float:
    return rating + k * multiplier * (actual - expected)

In [37]:
matches_fd = matches_fb.sort_values(['season', 'date']).reset_index(drop=True).copy()

matches_fd['home_elo_pre'] = np.nan
matches_fd['away_elo_pre'] = np.nan

home_teams = matches_fd['home_team'].values
away_teams = matches_fd['away_team'].values
seasons    = matches_fd['season'].values
ftrs       = matches_fd['ftr'].values
fthg_arr   = matches_fd['fthg'].values
ftag_arr   = matches_fd['ftag'].values

home_elo_arr = np.full(len(matches_fd), np.nan)
away_elo_arr = np.full(len(matches_fd), np.nan)

for season in sorted(np.unique(seasons)):
    season_idx = np.where(seasons == season)[0]
    
    equipos = set(home_teams[season_idx]) | set(away_teams[season_idx])
    elo = {team: ELO_INITIAL for team in equipos}
    
    for idx in season_idx:
        home = home_teams[idx]
        away = away_teams[idx]
        ftr  = ftrs[idx]
        goal_diff = fthg_arr[idx] - ftag_arr[idx]
        home_elo_arr[idx] = elo[home]
        away_elo_arr[idx] = elo[away]
        
        exp_home = expected_score(elo[home], elo[away], ELO_HOME_ADVANTAGE)
        exp_away = 1.0 - exp_home
        
        if ftr == 'H':
            actual_home, actual_away = 1.0, 0.0
        elif ftr == 'A':
            actual_home, actual_away = 0.0, 1.0
        else:
            actual_home, actual_away = 0.5, 0.5
        
        mult = margin_multiplier(goal_diff)
        
        elo[home] = update_elo(elo[home], actual_home, exp_home, ELO_K, mult)
        elo[away] = update_elo(elo[away], actual_away, exp_away, ELO_K, mult)

matches_fd['home_elo_pre'] = home_elo_arr
matches_fd['away_elo_pre'] = away_elo_arr
matches_fd['elo_diff'] = matches_fd['home_elo_pre'] - matches_fd['away_elo_pre']



## 5. Familia E — Mercado de apuestas

Esta familia incorpora una variable a partir de las casas de apuestas:

- `match_uncertainty`: variable derivada que cuantifica la incertidumbre competitiva del partido. Se calcula como `1 − max(prob_h, prob_d, prob_a)`. Valores próximos a cero indican partidos con un favorito claro y valores próximos a 2/3 ≈ 0,667 corresponden a partidos perfectamente equilibrados.

In [38]:
matches_fe = matches_fd.copy()


matches_fe['match_uncertainty'] = 1.0 - pd.concat(
    [matches_fe['prob_h'], matches_fe['prob_d'], matches_fe['prob_a']],
    axis=1
).max(axis=1)

## 6. Familia F — Contexto del partido

Variables que reflejan condiciones contextuales del enfrentamiento más allá del rendimiento histórico de los equipos:

- `home_rest_days`, `away_rest_days`: número de días transcurridos desde el último partido oficial de cada equipo.
- `rest_diff`: diferencia entre los descansos del local y el visitante. Valores positivos indican mayor descanso para el local.

In [39]:
long_with_rest = matches_long.copy()

long_with_rest['rest_days'] = (
    long_with_rest
    .groupby(['team', 'season'])['date']
    .transform(lambda s: s.diff().dt.days)
)

In [40]:
matches_ff = matches_fe.copy()

home_rest = long_with_rest[long_with_rest['is_home']][['season', 'date', 'team', 'rest_days']].copy()
home_rest = home_rest.rename(columns={'team': 'home_team', 'rest_days': 'home_rest_days'})

away_rest = long_with_rest[~long_with_rest['is_home']][['season', 'date', 'team', 'rest_days']].copy()
away_rest = away_rest.rename(columns={'team': 'away_team', 'rest_days': 'away_rest_days'})

matches_ff = matches_ff.merge(home_rest, on=['season', 'date', 'home_team'], how='left')
matches_ff = matches_ff.merge(away_rest, on=['season', 'date', 'away_team'], how='left')

matches_ff['rest_diff'] = matches_ff['home_rest_days'] - matches_ff['away_rest_days']

assert len(matches_ff) == len(matches_fe), "ALERTA: el merge ha duplicado filas"

## 7. Ensamblado y validación

Una vez construidas las familias de features, se realiza la selección final de variables que conformarán el dataset de modelado.

Se descartan explícitamente:

- `prob_d`: utilizada únicamente como insumo intermedio para construir `match_uncertainty`.
- Estadísticas de partido en bruto (tiros, faltas, córners, tarjetas) y cuotas brutas de B365, BW y WH: solo se conservan las probabilidades calibradas y las features derivadas de ellas.
- Identificadores auxiliares (`season_label`, `home_match_n`, `away_match_n`): no aportan al modelo y reducen ruido en el dataset.

In [ ]:
COLUMNAS_FINALES = {
    'identificadores': [
        'season', 'date', 'jornada', 'home_team', 'away_team',
    ],
    'family_a_rolling': [
        'home_ppg_3', 'away_ppg_3', 'home_gf_3', 'away_gf_3', 'home_gc_3', 'away_gc_3',
        'home_ppg_5', 'away_ppg_5', 'home_gf_5', 'away_gf_5', 'home_gc_5', 'away_gc_5',
        'home_ppg_10', 'away_ppg_10', 'home_gf_10', 'away_gf_10', 'home_gc_10', 'away_gc_10',
    ],
    'family_b_clasificacion': [
        'home_pos_pre', 'away_pos_pre',
        'home_pts_pre', 'away_pts_pre',
        'home_gd_pre', 'away_gd_pre',
    ],
    'family_c_valor': [
        'home_mv', 'away_mv',
    ],
    'family_d_elo': [
        'home_elo_pre', 'away_elo_pre',
    ],
    'family_e_mercado': [
        'prob_h', 'match_uncertainty',
    ],
    'family_f_contexto': [
        'home_rest_days', 'away_rest_days',
    ],
    'target_y_goles': [
        'ftr', 'fthg', 'ftag',
    ],
}

columnas_a_conservar = [col for sublist in COLUMNAS_FINALES.values() for col in sublist]

faltan_cols = [c for c in columnas_a_conservar if c not in matches_ff.columns]
assert not faltan_cols, f"Columnas que faltan en matches_ff: {faltan_cols}"

dataset_final = matches_ff[columnas_a_conservar].copy()

print(f"Dataset final: {len(dataset_final):,} filas × {len(dataset_final.columns)} columnas\n")
print(f"Distribución de columnas por familia:")
for familia, cols in COLUMNAS_FINALES.items():
    print(f"  {familia:<25} {len(cols):>3} columnas")
print(f"  {'TOTAL':<25} {len(columnas_a_conservar):>3} columnas")

Dataset final: 7,295 filas × 40 columnas

Distribución de columnas por familia:
  identificadores             5 columnas
  family_a_rolling           18 columnas
  family_b_clasificacion      6 columnas
  family_c_valor              2 columnas
  family_d_elo                2 columnas
  family_e_mercado            2 columnas
  family_f_contexto           2 columnas
  target_y_goles              3 columnas
  TOTAL                      40 columnas


In [ ]:
print("Test global anti-leakage:")
print("Correlación entre cada feature numérica y el resultado (codificado H=1, D=0, A=-1):\n")

ftr_num = dataset_final['ftr'].map({'H': 1, 'D': 0, 'A': -1})
features_numericas = [
    c for c in dataset_final.columns 
    if c not in ['season', 'date', 'jornada', 'home_team', 'away_team', 'ftr', 'fthg', 'ftag']
]

correlaciones = {}
for col in features_numericas:
    correlaciones[col] = abs(dataset_final[col].corr(ftr_num))

top = sorted(correlaciones.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top 10 correlaciones |feature ~ resultado|:")
for col, corr in top:
    flag = "⚠" if corr > 0.5 else "✓"
    print(f"  {flag} {col:<25} {corr:.4f}")

print("\nValores > 0,5 indicarían posible leakage. Esperado: todas < 0,45.")

Test global anti-leakage:
Correlación entre cada feature numérica y el resultado (codificado H=1, D=0, A=-1):

Top 10 correlaciones |feature ~ resultado|:
  ✓ prob_h                    0.2361
  ✓ match_uncertainty         0.1811
  ✓ away_gd_pre               0.1240
  ✓ away_elo_pre              0.1169
  ✓ home_gd_pre               0.1157
  ✓ home_elo_pre              0.1130
  ✓ home_mv                   0.1087
  ✓ away_mv                   0.1083
  ✓ away_pos_pre              0.1029
  ✓ home_pos_pre              0.1003

Valores > 0,5 indicarían posible leakage. Esperado: todas < 0,45.


In [43]:
print(f"\nNaN en columnas críticas:")
for col in ['ftr', 'fthg', 'ftag', 'season', 'date', 'home_team', 'away_team']:
    n = dataset_final[col].isna().sum()
    status = "✓" if n == 0 else "✗"
    print(f"  {status} {col:<15} NaN: {n}")


NaN en columnas críticas:
  ✓ ftr             NaN: 0
  ✓ fthg            NaN: 0
  ✓ ftag            NaN: 0
  ✓ season          NaN: 0
  ✓ date            NaN: 0
  ✓ home_team       NaN: 0
  ✓ away_team       NaN: 0


## 8. Partición temporal

Aplicación de la partición temporal.

- **train**: 2010-11 a 2019-20
- **validation**: 2020-21 a 2022-23
- **test**: 2023-24 a 2024-25
- **demo**: 2025-26 (temporada en curso)

In [44]:
def asignar_split(season):
    for nombre, temporadas in SPLITS.items():
        if season in temporadas:
            return nombre
    return None

dataset_final['split'] = dataset_final['season'].apply(asignar_split)
sin_split = dataset_final['split'].isna().sum()
assert sin_split == 0, f"Hay {sin_split} filas sin split asignado"

print("Distribución por partición:")
for split in ['train', 'validation', 'test', 'demo']:
    subset = dataset_final[dataset_final['split'] == split]
    if len(subset) > 0:
        print(f"  {split:<12} {len(subset):>5,} filas  | "
              f"fechas: {subset['date'].min().date()} → {subset['date'].max().date()}  | "
              f"{subset['season'].nunique()} temporadas")

train_max = dataset_final[dataset_final['split'] == 'train']['date'].max()
val_min   = dataset_final[dataset_final['split'] == 'validation']['date'].min()
val_max   = dataset_final[dataset_final['split'] == 'validation']['date'].max()
test_min  = dataset_final[dataset_final['split'] == 'test']['date'].min()

assert train_max < val_min, f"LEAKAGE: train termina en {train_max}, validation empieza en {val_min}"
assert val_max < test_min,  f"LEAKAGE: validation termina en {val_max}, test empieza en {test_min}"
print("\n✓ Partición temporal sin fugas entre train, validation y test")

Distribución por partición:
  train        4,592 filas  | fechas: 2010-08-27 → 2020-08-07  | 10 temporadas
  validation   1,373 filas  | fechas: 2020-09-12 → 2023-05-28  | 3 temporadas
  test           923 filas  | fechas: 2023-08-11 → 2025-06-01  | 2 temporadas
  demo           407 filas  | fechas: 2025-08-15 → 2026-04-27  | 1 temporadas

✓ Partición temporal sin fugas entre train, validation y test


## 9. Persistencia del dataset

Guardado del dataset final en formato Parquet en `data/processed/dataset_modelado.parquet`.

In [45]:
columnas_orden = (
    ['split'] +
    COLUMNAS_FINALES['identificadores'] +
    COLUMNAS_FINALES['family_a_rolling'] +
    COLUMNAS_FINALES['family_b_clasificacion'] +
    COLUMNAS_FINALES['family_c_valor'] +
    COLUMNAS_FINALES['family_d_elo'] +
    COLUMNAS_FINALES['family_e_mercado'] +
    COLUMNAS_FINALES['family_f_contexto'] +
    COLUMNAS_FINALES['target_y_goles']
)
dataset_final = dataset_final[columnas_orden]

output_path = PROCESSED_DIR / "dataset_modelado.parquet"
dataset_final.to_parquet(output_path, index=False)

## 10. Resumen del dataset construido

Tabla descriptiva de las 49 variables del dataset final, organizadas por familia. Esta tabla se incluirá en el apartado de Construcción del Dataset de la memoria (capítulo 4).

In [46]:
descripciones = {
    'split':            'Partición temporal (train/validation/test/demo)',
    'season':           'Año de inicio de la temporada',
    'date':             'Fecha del partido',
    'jornada':          'Jornada lógica cronológica (sección 4.4 de la memoria)',
    'home_team':        'Nombre canónico del equipo local',
    'away_team':        'Nombre canónico del equipo visitante',
    'home_ppg_3':       'Puntos por partido del local, últimos 3 partidos',
    'away_ppg_3':       'Puntos por partido del visitante, últimos 3 partidos',
    'home_gf_3':        'Goles a favor del local, últimos 3 partidos',
    'away_gf_3':        'Goles a favor del visitante, últimos 3 partidos',
    'home_gc_3':        'Goles en contra del local, últimos 3 partidos',
    'away_gc_3':        'Goles en contra del visitante, últimos 3 partidos',
    'home_ppg_5':       'Puntos por partido del local, últimos 5 partidos',
    'away_ppg_5':       'Puntos por partido del visitante, últimos 5 partidos',
    'home_gf_5':        'Goles a favor del local, últimos 5 partidos',
    'away_gf_5':        'Goles a favor del visitante, últimos 5 partidos',
    'home_gc_5':        'Goles en contra del local, últimos 5 partidos',
    'away_gc_5':        'Goles en contra del visitante, últimos 5 partidos',
    'home_ppg_10':      'Puntos por partido del local, últimos 10 partidos',
    'away_ppg_10':      'Puntos por partido del visitante, últimos 10 partidos',
    'home_gf_10':       'Goles a favor del local, últimos 10 partidos',
    'away_gf_10':       'Goles a favor del visitante, últimos 10 partidos',
    'home_gc_10':       'Goles en contra del local, últimos 10 partidos',
    'away_gc_10':       'Goles en contra del visitante, últimos 10 partidos',
    'home_pos_pre':     'Posición clasificatoria del local antes del partido',
    'away_pos_pre':     'Posición clasificatoria del visitante antes del partido',
    'home_pts_pre':     'Puntos acumulados del local antes del partido',
    'away_pts_pre':     'Puntos acumulados del visitante antes del partido',
    'home_gd_pre':      'Diferencia de goles acumulada del local antes del partido',
    'away_gd_pre':      'Diferencia de goles acumulada del visitante antes del partido',
    'home_mv':          'Valor de mercado de la plantilla del local (M€)',
    'away_mv':          'Valor de mercado de la plantilla del visitante (M€)',
    'home_elo_pre':     'Rating Elo del local antes del partido',
    'away_elo_pre':     'Rating Elo del visitante antes del partido',
    'prob_h':           'Probabilidad implícita de victoria local según mercado 1X2',
    'match_uncertainty':'Incertidumbre competitiva: 1 − max(prob_h, prob_d, prob_a)',
    'home_rest_days':   'Días desde el último partido del local',
    'away_rest_days':   'Días desde el último partido del visitante',
    'ftr':              'Resultado del partido: H (local), D (empate), A (visitante)',
    'fthg':             'Goles del equipo local',
    'ftag':             'Goles del equipo visitante',
}

familias_legibles = {
    'identificadores':         'ID',
    'family_a_rolling':        'A · Forma reciente',
    'family_b_clasificacion':  'B · Clasificación',
    'family_c_valor':          'C · Valor mercado',
    'family_d_elo':            'D · Elo',
    'family_e_mercado':        'E · Mercado apuestas',
    'family_f_contexto':       'F · Contexto',
    'target_y_goles':          'Target',
}

filas = []
filas.append({'variable': 'split', 'familia': 'ID', 'descripcion': descripciones['split']})
for familia, cols in COLUMNAS_FINALES.items():
    for col in cols:
        filas.append({
            'variable': col,
            'familia': familias_legibles[familia],
            'descripcion': descripciones.get(col, '(sin descripción)'),
        })

resumen = pd.DataFrame(filas)

print(f"Resumen del dataset: {len(resumen)} variables\n")
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 80):
    print(resumen.to_string(index=False))

resumen_path = PROCESSED_DIR / "resumen_variables.csv"
resumen.to_csv(resumen_path, index=False)
print(f"\n✓ Tabla resumen guardada en: {resumen_path}")

Resumen del dataset: 41 variables

         variable              familia                                                   descripcion
            split                   ID               Partición temporal (train/validation/test/demo)
           season                   ID                                 Año de inicio de la temporada
             date                   ID                                             Fecha del partido
          jornada                   ID        Jornada lógica cronológica (sección 4.4 de la memoria)
        home_team                   ID                              Nombre canónico del equipo local
        away_team                   ID                          Nombre canónico del equipo visitante
       home_ppg_3   A · Forma reciente              Puntos por partido del local, últimos 3 partidos
       away_ppg_3   A · Forma reciente          Puntos por partido del visitante, últimos 3 partidos
        home_gf_3   A · Forma reciente                  